In [1]:
import sys
sys.path.append(r"C:\Users\Fonta\IdeaProjects\ATI_Analysis")

import ipywidgets as widgets
from IPython.display import display
from app.database.queries.evidence.read import get_all_academic_years, find_year_success_evidence_by_academic_year
from app.database.queries.tools.utilities import get_all_implementations
from app.database.queries.evidence.update import assign_implementation_to_year_success_indicator
from app.database.queries.evidence.read import get_yses_by_year_and_implementation

# Create the dropdowns
academic_year_dropdown = widgets.Dropdown(
    options=[],
    description='Academic Year:',
    layout=widgets.Layout(width='500px')
)

year_success_evidences_dropdown = widgets.Dropdown(
    options=[],
    description='Year Success Evidences:',
    layout=widgets.Layout(width='500px')
)

# Create a Textarea widget to display the success_indicator text
success_indicator_textarea = widgets.Textarea(
    value='',
    placeholder='Success Indicator will be displayed here',
    description='Success Indicator:',
    layout=widgets.Layout(width='500px')
)

# Create a Textarea widget to display the output of get_yses_by_year_and_implementation
output_textarea = widgets.Textarea(
    value='',
    placeholder='Output will be displayed here...',
    description='Output:',
    layout=widgets.Layout(width='500px', height='200px'),
    disabled=True
)

# Initialize a global variable to store year success evidences
year_success_evidences = []

def on_academic_year_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        update_year_success_evidences_dropdown(change['new'])

def update_year_success_evidences_dropdown(academic_year):
    global year_success_evidences
    year_success_evidences = find_year_success_evidence_by_academic_year(academic_year)
    year_success_evidences_dropdown.options = [(evidence['year_identifier'], evidence['year_identifier']) for evidence in year_success_evidences]

def on_year_success_evidence_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        # Find the selected year_success_evidence
        selected_year_success_evidence = next((evidence for evidence in year_success_evidences if evidence['year_identifier'] == change['new']), None)
        if selected_year_success_evidence:
            # Update the Textarea widget with the success_indicator text
            success_indicator_textarea.value = selected_year_success_evidence['success_indicator']
        else:
            success_indicator_textarea.value = ''

def populate_academic_year_dropdown():
    academic_years = get_all_academic_years()
    options = [(year.name, year.name) for year in academic_years]
    academic_year_dropdown.options = options

# Create the dropdown for implementation
implementation_dropdown = widgets.Dropdown(
    options=['process', 'project', 'procedure', 'service', 'guideline'],
    value='process',
    description='Implementation:',
    layout=widgets.Layout(width='500px')
)

# Define a new dropdown widget for implementation outputs
implementation_output_dropdown = widgets.Dropdown(
    options=[],
    description='Implementation Output:',
    layout=widgets.Layout(width='500px')
)

# Create a button widget
assign_button = widgets.Button(
    description='Assign Implementation',
    button_style='success',  # 'success', 'info', 'warning', 'danger' or ''
)

# Define a function that will be triggered when the button is clicked
def on_assign_button_clicked(button):
    # Retrieve the current values of the dropdowns
    year_success_identifier = year_success_evidences_dropdown.value
    implementation_type = implementation_dropdown.value
    implementation_title = implementation_output_dropdown.value
    academic_year = academic_year_dropdown.value

    output = get_yses_by_year_and_implementation(academic_year, implementation_type, implementation_title)
    output_textarea.value = '\n'.join([item["year_identifier"] for item in output])
    # Call assign_implementation_to_year_success_indicator with the retrieved values

    assign_implementation_to_year_success_indicator(year_success_identifier, implementation_type, implementation_title)

# Attach the on_assign_button_clicked function to the button
assign_button.on_click(on_assign_button_clicked)

# Define a function that will be triggered when the implementation_dropdown value changes
def on_implementation_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        # Call get_all_implementations with the new value of implementation_dropdown
        implementations = get_all_implementations(change['new'])
        # Update the options of implementation_output_dropdown
        implementation_output_dropdown.options = [(imp.title, imp.title) for imp in implementations]

# Attach the on_implementation_change function to implementation_dropdown
implementation_dropdown.observe(on_implementation_change, names='value')

# Define a function that will be triggered when the implementation_output_dropdown value changes
def on_implementation_output_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        # Call get_yses_by_year_and_implementation and update the output_textarea
        implementation_type = implementation_dropdown.value
        implementation_title = implementation_output_dropdown.value
        academic_year = academic_year_dropdown.value

        output = get_yses_by_year_and_implementation(academic_year, implementation_type, implementation_title)
        output_textarea.value = '\n'.join([f"{item['year_identifier']} - {item['status_level']}" for item in output])

# Attach the on_implementation_output_change function to implementation_output_dropdown
implementation_output_dropdown.observe(on_implementation_output_change, names='value')

# Populate the academic year dropdown and set up the observer
populate_academic_year_dropdown()
academic_year_dropdown.observe(on_academic_year_change, names='value')
year_success_evidences_dropdown.observe(on_year_success_evidence_change, names='value')

# Create a VBox container for the dropdowns
dropdowns_container = widgets.VBox([
    academic_year_dropdown,
    year_success_evidences_dropdown,
    implementation_dropdown,
    implementation_output_dropdown
])

# Create a VBox container for the Textarea widgets
textarea_container = widgets.VBox([success_indicator_textarea, output_textarea])

# Create a VBox container to display the dropdowns, Textarea widgets, and button
layout_container = widgets.VBox([
    widgets.HBox([dropdowns_container, textarea_container]),
    assign_button
])

# Display the layout container
display(layout_container)
